# 02 — Conventional laser-welding data audit and cleaning

This notebook works on a copy of `data/raw/Laser Welding.xlsx`. It preserves the source workbook and does not automatically delete rows, replace missing values, or convert zero measurements into missing values.

First we create a transparent audit. Cleaning decisions will be made only after reviewing the actual columns and records.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "Laser Welding.xlsx"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_FILE.exists(), f"Cannot find source workbook: {RAW_FILE}"
print("Source workbook:", RAW_FILE)
print("Output folder:", OUT_DIR)

## 1. Load every sheet without changing the source

In [ ]:
source_book = pd.ExcelFile(RAW_FILE)
print("Sheets:", source_book.sheet_names)
raw_sheets = {name: pd.read_excel(RAW_FILE, sheet_name=name) for name in source_book.sheet_names}
for name, frame in raw_sheets.items():
    print(f"\n--- {name}: {frame.shape[0]} rows × {frame.shape[1]} columns ---")
    display(frame.head(10))

## 2. Audit column names, data types, missing values and duplicates

This report is diagnostic. Duplicate rows are flagged for review, not removed.

In [ ]:
audit_rows = []
for sheet_name, frame in raw_sheets.items():
    for column in frame.columns:
        audit_rows.append({
            "sheet": sheet_name,
            "column": str(column),
            "dtype": str(frame[column].dtype),
            "non_missing": int(frame[column].notna().sum()),
            "missing": int(frame[column].isna().sum()),
            "unique_non_missing": int(frame[column].nunique(dropna=True))
        })
    print(f"\n{sheet_name}: exact duplicate rows = {int(frame.duplicated().sum())}")
column_audit = pd.DataFrame(audit_rows)
display(column_audit)

## 3. Flag zero-valued numeric measurements for investigation

A zero can mean a genuine measured zero, no weld, lack of fusion, or a placeholder. We must use the source's experimental meaning before deciding how to treat it.

In [ ]:
zero_reports = []
for sheet_name, frame in raw_sheets.items():
    numeric = frame.select_dtypes(include="number")
    for column in numeric.columns:
        count = int((numeric[column] == 0).sum())
        if count:
            zero_reports.append({"sheet": sheet_name, "column": str(column), "zero_count": count})
zero_audit = pd.DataFrame(zero_reports, columns=["sheet", "column", "zero_count"])
display(zero_audit if not zero_audit.empty else pd.DataFrame({"result": ["No numeric zero values found"]}))

## 4. Create an untouched copy for the next cleaning stage

This export preserves all rows and values. It is a checkpoint, not yet the final cleaned dataset.

In [ ]:
checkpoint = OUT_DIR / "conventional_laser_welding_audit_copy.xlsx"
with pd.ExcelWriter(checkpoint, engine="openpyxl") as writer:
    for sheet_name, frame in raw_sheets.items():
        frame.to_excel(writer, sheet_name=str(sheet_name)[:31], index=False)
    column_audit.to_excel(writer, sheet_name="Column_Audit", index=False)
    zero_audit.to_excel(writer, sheet_name="Zero_Value_Audit", index=False)
print("Saved audit copy:", checkpoint)
print("Original file remains unchanged:", RAW_FILE.exists())

## 5. Stop and review before applying transformations

Do not drop rows or fill missing values yet. Confirm the meaning of zero-valued outputs and identify which columns are inputs, outputs, specimen identifiers, and repeat numbers. The next cleaning revision will standardize the confirmed fields and document each transformation.